# Agent Repair Pipeline — MuSiQue

**ICLR Experiment Pipeline — Colab A100-80GB + Qwen2.5-32B-Instruct-AWQ**

| Stage | What it does | ~Time (A100-80GB, 500 Qs) |
|---|---|---|
| 0 | Download MuSiQue, create folders | 1 min |
| 1 | ReAct trajectory generation (Qwen2.5-32B-Instruct-AWQ) | ~1-2 h |
| 2 | Step-level uncertainty scoring | ~1-2 h |
| 3 | 72B judge error annotation | ~1-2 h |
| 4 | Localization scoring (+ cascade-aware rules) | < 1 min |
| 5 | 126 repair strategies (backtrack × nudge type ablation) | ~4-6 h |
| 6 | Tables, stats, figures, ablation analysis | < 1 min |

**Every stage is resumable** — if Colab disconnects, re-run the cell and it picks up where it left off.

---
## 0. Setup

In [ ]:
# Verify GPU — should show A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_musique.yaml'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kishormorol/agent-repair.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Check Colab's CUDA version first
!nvcc --version 2>/dev/null || echo "nvcc not found"
!python -c "import torch; print(f'torch CUDA: {torch.version.cuda}')" 2>/dev/null || echo "torch not installed yet"
!nvidia-smi | grep "CUDA Version"

In [ ]:
# Install vLLM — pinned to 0.19.0 (last version with CUDA 12 default wheels)
# vLLM >= 0.20 ships CUDA 13 wheels which don't work on Colab's CUDA 12.x
!pip uninstall -y vllm 2>/dev/null
!pip install -q "vllm==0.19.0" 2>&1 | tail -5

# Install remaining dependencies
!pip install -q "transformers>=4.57.0" "tokenizers>=0.21.0" accelerate \
    huggingface_hub datasets scipy scikit-learn pandas pyarrow \
    pyyaml tqdm matplotlib seaborn statsmodels 2>&1 | tail -3

print('\n--- Installation complete ---')

In [ ]:
# Quick sanity check: can vLLM see the GPU?
import torch, vllm
print(f'torch:  {torch.__version__}')
print(f'vLLM:   {vllm.__version__}')
print(f'CUDA:   {torch.version.cuda}')
print(f'GPU:    {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Clear previous data
import shutil, os
drive_base = '/content/drive/MyDrive/agent-repair-musique'
for d in ['outputs', 'data/processed']:
    p = os.path.join(drive_base, d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
for d in ['outputs', 'data/processed']:
    p = os.path.join('/content/agent-repair', d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
print('Ready for fresh run with MuSiQue')

---
## Smoke Test (20 questions, ~10-15 min)

**Run this first** to verify everything works before committing A100 units to the full 500-question run.

In [ ]:
stages = ['run_setup', 'run_generate', 'run_uncertainty', 'run_annotate',
          'run_localize', 'run_repair', 'run_eval']

for stage in stages:
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG} --limit 20
    print()

In [ ]:
# Check smoke test produced output
import glob
tables = glob.glob('outputs/tables/*') + glob.glob('/content/drive/MyDrive/agent-repair-musique/outputs/tables/*')
figs = glob.glob('outputs/figures/*') + glob.glob('/content/drive/MyDrive/agent-repair-musique/outputs/figures/*')
print(f'Tables: {len(tables)}, Figures: {len(figs)}')
if tables or figs:
    print('Smoke test PASSED — safe to run the full pipeline below.')
else:
    print('WARNING: No outputs found. Check the logs above for errors.')

---
## Full Pipeline (500 questions)

Only run these cells after the smoke test passes. To reset and run fresh, delete the `data/processed/` folder.

### Stage 0: Download MuSiQue

In [ ]:
!python scripts/run_setup.py --config {CONFIG}

### Stage 1: Generate ReAct Trajectories

In [ ]:
!python scripts/run_generate.py --config {CONFIG}

### Stage 2: Step-Level Uncertainty

In [ ]:
!python scripts/run_uncertainty.py --config {CONFIG}

### Stage 3: Error Annotation (72B Judge)

In [ ]:
!python scripts/run_annotate.py --config {CONFIG}

### Stage 4: Localization Scoring

In [ ]:
!python scripts/run_localize.py --config {CONFIG}

### Stage 5: Repair (126 Strategies)

In [ ]:
!python scripts/run_repair.py --config {CONFIG}

### Stage 6: Evaluation

In [ ]:
!python scripts/run_eval.py --config {CONFIG}

---
## Results

In [ ]:
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print('Summary not yet generated. Run Stage 6 first.')

In [ ]:
import pandas as pd

results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)
else:
    print('Results table not yet generated. Run Stage 6 first.')

In [ ]:
import glob
from IPython.display import display, Image

fig_dir = cfg.path('figures')
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))
if figs:
    for f in figs:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(filename=f, width=800))
else:
    print('No figures yet. Run Stage 6 first.')

---
## Re-run Stages 4–6 with Cascade-Aware Strategies

**Use this after you already have stages 1–3 completed** (trajectories, uncertainty, annotations).
This pulls the latest code (with cascade rules), clears only localization/repair/eval outputs,
and re-runs stages 4→5→6 in one cell. ~3-4 hours on A100-80GB for 500 questions.

In [ ]:
import os, shutil

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_musique.yaml'

# 1. Pull latest code with cascade-aware strategies
os.chdir(REPO_DIR)
!git pull

# 2. Clear ONLY stages 4-6 outputs (keep trajectories, uncertainty, annotations)
drive_base = '/content/drive/MyDrive/agent-repair-musique'
for base in [REPO_DIR, drive_base]:
    for d in ['outputs/localization', 'outputs/repairs', 'outputs/tables', 'outputs/figures']:
        p = os.path.join(base, d)
        if os.path.exists(p):
            shutil.rmtree(p)
            print(f'Cleared {p}')

# 3. Clear checkpoint files for stages 4-6 (these live in outputs/logs/)
for base in [REPO_DIR, drive_base]:
    for ckpt in ['outputs/logs/stage4_localize.jsonl',
                 'outputs/logs/stage5_repair.jsonl',
                 'outputs/logs/stage6_eval.jsonl']:
        p = os.path.join(base, ckpt)
        if os.path.exists(p):
            os.remove(p)
            print(f'Removed checkpoint: {p}')

print('\nStages 1-3 data preserved. Ready to re-run 4→5→6.')

In [ ]:
# Run stages 4 → 5 → 6 sequentially (run this and go to sleep)
import time

for stage in ['run_localize', 'run_repair', 'run_eval']:
    t0 = time.time()
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG}
    elapsed = (time.time() - t0) / 60
    print(f'  ✓ {stage} done in {elapsed:.1f} min\n')

print('\n' + '='*60)
print('  ALL DONE — check results below')
print('='*60)

In [ ]:
# View cascade-aware results
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())

import pandas as pd
results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)

import glob
from IPython.display import display, Image
figs = sorted(glob.glob(os.path.join(cfg.path('figures'), '*.png')))
for f in figs:
    print(f'\n--- {os.path.basename(f)} ---')
    display(Image(filename=f, width=800))